# PoliMillionaire History Chatbot

This notebook is a history-only text chatbot for **Who wants to be a PoliMillionaire?**. It uses the provided `millionaire_client` package and plays only `comp_id = 1`.

Design constraints from the assignment:

- each question must be answered inside about 30 seconds;
- the answering model must run locally, not through a paid or hosted LLM API;
- open-weight models are used;
- RAG is allowed when the external source returns raw, non-generated content;
- the backend/client package is not modified;
- no voice interface is used in this version.

The solution uses a local Qwen2.5-1.5B-Instruct model plus a Wikipedia RAG retriever. Wikipedia is accessed through the free MediaWiki API and only raw encyclopedia text is used.

Coding assistant statement: Codex helped scaffold this notebook. The group should review, understand, run, evaluate, and explain every part before submission.

## 1. Install Dependencies

This notebook is intended for Google Colab with a T4 GPU. The answerer uses `Qwen/Qwen2.5-1.5B-Instruct` loaded locally in 4-bit through `AutoModelForCausalLM`.


In [77]:
import importlib.util
import subprocess
import sys

requirements = [
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("rank-bm25", "rank_bm25"),
    ("scikit-learn", "sklearn"),
    ("sentence-transformers", "sentence_transformers"),
    ("transformers", "transformers"),
]

missing = [package for package, module in requirements if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Environment ready")


Environment ready


## 2. Connect the Assignment Client

Place the `millionaire_client` folder in your Google Drive assignment directory, as in the tutorial notebook.

In [78]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/gdrive/")
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

candidate_package_dirs = [
    Path("/content/gdrive/MyDrive/NLP_assignment"),
    Path("/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment"),
    Path.cwd(),
]

for package_dir in candidate_package_dirs:
    if (package_dir / "millionaire_client").exists():
        if str(package_dir) not in sys.path:
            sys.path.append(str(package_dir))
        print(f"Using package path: {package_dir}")
        break
else:
    raise FileNotFoundError("Could not find millionaire_client. Check your Drive path.")

from millionaire_client import MillionaireClient, AuthenticationError

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
Using package path: /content/gdrive/MyDrive/NLP_assignment


## 3. Login and Select History

Use Colab Secrets with key `poli-millionaire`, or enter the password when prompted. The password is intentionally not stored in the notebook.

In [ ]:
import getpass

API_URL = "http://131.175.15.22:51111/"
COMP_ID = 1
username = ""

password = os.environ.get("POLI_MILLIONAIRE_PASSWORD", "")
try:
    from google.colab import userdata
    password = userdata.get("poli-millionaire") or password
except Exception:
    pass

if not password:
    password = getpass.getpass("PoliMillionaire password: ")

client = MillionaireClient(API_URL, timeout=10)
try:
    user = client.login(username, password)
    print(f"Welcome, {user.username}! Role: {user.role}")
except AuthenticationError as exc:
    raise RuntimeError(f"Login failed: {exc}")

print("Available competitions")
for comp in client.competitions.list_all():
    marker = " <- selected" if comp.id == COMP_ID else ""
    print(f"{comp.id}: {comp.name} ({comp.max_levels} questions){marker}")

history_config = client.competitions.get_config(COMP_ID)
print(f"Selected competition: {history_config.id} - {history_config.name}")

Welcome, gary! Role: student
Available competitions
0: Entertainment (15 questions)
1: Ancient History and Politics (15 questions) <- selected
2: Science and Nature (15 questions)
3: Maths (15 questions)
Selected competition: 1 - Ancient History and Politics


## 4. Load Local Models

The dense retriever uses MiniLM embeddings. The answer selector uses Qwen2.5-1.5B-Instruct, a local open-weight causal instruction model loaded with 4-bit quantization on T4. No LLM API is called.


In [80]:
import gc
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

QUESTION_SECONDS = 30
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_MAX_INPUT_TOKENS = 1400

if "llm_model" in globals():
    del llm_model
if "embedding_model" in globals():
    del embedding_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Device: {DEVICE}")
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

print(f"Loading local causal instruction model: {LLM_MODEL_NAME}")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

if DEVICE == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
    llm_model.to(DEVICE)

llm_model.eval()
print("Local models ready")


Device: cuda
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading local causal instruction model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Local models ready


## 5. Wikipedia RAG Retrieval

For each question, the retriever builds search queries from the question and answer options, downloads raw article text from Wikipedia, splits it into chunks, and ranks chunks using BM25 plus embedding similarity.

In [81]:
import json
import math
import re
import time
from dataclasses import dataclass
from functools import lru_cache
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
from rank_bm25 import BM25Okapi

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "there", "these", "this", "those", "to", "was", "were",
    "what", "when", "where", "which", "who", "why", "with", "during", "did",
    "do", "does", "according", "following", "among", "between", "into", "than",
    "then", "through", "under", "over", "after", "before", "period", "status",
    "purpose", "reason", "used", "use", "known", "called", "name", "term",
    "main", "primary", "first", "last", "most", "least"
}

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "PoliMillionaireHistoryRAG/1.0 raw Wikipedia retrieval"

@dataclass
class RetrievedChunk:
    title: str
    text: str
    score: float


def option_text(option):
    return option.text if hasattr(option, "text") else option["text"]


def option_id(option):
    return option.id if hasattr(option, "id") else option["id"]


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).replace("\xa0", " ")).strip()


def clean_wiki_text(text: str) -> str:
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\{\{.*?\}\}", " ", text, flags=re.DOTALL)
    text = re.sub(r"<ref[^>]*>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\([^)]*(?:listen|pronunciation|help·info|born|died)[^)]*\)", " ", text, flags=re.IGNORECASE)
    section_stop = re.search(r"\n\s*(?:See also|References|Bibliography|Further reading|External links|Notes|Sources)\s*\n", text, flags=re.IGNORECASE)
    if section_stop:
        text = text[:section_stop.start()]
    cleaned_lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if re.fullmatch(r"[A-Z][A-Za-z ]{1,40}", line):
            continue
        if len(line) < 35 and not re.search(r"[.!?]$", line):
            continue
        cleaned_lines.append(line)
    return normalize_text(" ".join(cleaned_lines))


def tokenize(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if len(token) > 1 and token not in STOPWORDS]


def extract_keywords(text: str, limit: int = 12) -> list[str]:
    seen = set()
    keywords = []
    for token in tokenize(text):
        if token not in seen:
            keywords.append(token)
            seen.add(token)
        if len(keywords) >= limit:
            break
    return keywords


def build_search_queries(question_text: str, options: list[str]) -> list[str]:
    capital_phrases = re.findall(r"(?:[A-Z][a-z0-9]+(?:\s+[A-Z][a-z0-9]+)*)", question_text)
    question_keywords = extract_keywords(question_text, limit=14)
    option_keywords = extract_keywords(" ".join(options), limit=8)
    queries = []

    if capital_phrases:
        queries.append(" ".join(capital_phrases[:4]))
    if question_keywords:
        queries.append(" ".join(question_keywords[:10]))
    if question_keywords and option_keywords:
        queries.append(" ".join(question_keywords[:7] + option_keywords[:3]))
    queries.append(question_text)

    deduped = []
    seen = set()
    for query in queries:
        cleaned = normalize_text(query).lower()
        if cleaned and cleaned not in seen:
            deduped.append(query)
            seen.add(cleaned)
    return deduped[:4]


def wikipedia_request(params: dict, timeout: float = 4.0) -> dict:
    url = f"{WIKIPEDIA_API}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def search_wikipedia(query: str, limit: int = 4, timeout: float = 4.0) -> list[str]:
    data = wikipedia_request(
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": limit,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
    )
    return [item["title"] for item in data.get("query", {}).get("search", [])]


@lru_cache(maxsize=256)
def fetch_wikipedia_extract(title: str, timeout: float = 4.0) -> str:
    data = wikipedia_request(
        {
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "exsectionformat": "plain",
            "titles": title,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
    )
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return clean_wiki_text(page.get("extract", ""))
    return ""


def split_into_chunks(text: str, title: str, chunk_size: int = 850, overlap: int = 120) -> list[dict]:
    text = clean_wiki_text(text)
    if not text:
        return []
    chunks = []
    step = max(1, chunk_size - overlap)
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if len(chunk) >= 120:
            chunks.append({"title": title, "text": chunk})
        if start + chunk_size >= len(text):
            break
    return chunks


def collect_history_documents(question_text: str, options: list[str], max_pages: int = 3, deadline=None) -> list[dict]:
    titles = []
    seen_titles = set()
    for query in build_search_queries(question_text, options):
        if deadline and time.monotonic() > deadline - 9:
            break
        try:
            for title in search_wikipedia(query, limit=4):
                title_key = title.lower()
                if title_key not in seen_titles:
                    titles.append(title)
                    seen_titles.add(title_key)
                if len(titles) >= max_pages:
                    break
        except Exception as exc:
            print(f"Wikipedia search skipped for '{query}': {exc}")
        if len(titles) >= max_pages:
            break

    documents = []
    for title in titles[:max_pages]:
        if deadline and time.monotonic() > deadline - 7:
            break
        try:
            text = fetch_wikipedia_extract(title)
            if text:
                documents.append({"title": title, "text": text})
        except Exception as exc:
            print(f"Wikipedia page skipped for '{title}': {exc}")
    return documents

In [82]:
class HybridRetriever:
    def __init__(self, documents: list[dict], dense_model):
        self.chunks = []
        for document in documents:
            self.chunks.extend(split_into_chunks(document["text"], document["title"]))
        self.texts = [chunk["text"] for chunk in self.chunks]
        self.tokens = [tokenize(text) for text in self.texts]
        self.bm25 = BM25Okapi(self.tokens) if self.tokens else None
        self.dense_model = dense_model
        self.embeddings = None
        if self.texts:
            self.embeddings = self.dense_model.encode(
                self.texts,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

    @staticmethod
    def scale(scores: np.ndarray) -> np.ndarray:
        if scores.size == 0:
            return scores
        low = float(np.min(scores))
        high = float(np.max(scores))
        if math.isclose(low, high):
            return np.zeros_like(scores, dtype=float)
        return (scores - low) / (high - low)

    def retrieve(self, query: str, top_k: int = 5) -> list[RetrievedChunk]:
        if not self.texts:
            return []

        query_tokens = tokenize(query)
        bm25_scores = np.array(self.bm25.get_scores(query_tokens), dtype=float) if self.bm25 else np.zeros(len(self.texts))
        query_embedding = self.dense_model.encode([query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
        dense_scores = self.embeddings @ query_embedding
        combined = 0.42 * self.scale(bm25_scores) + 0.58 * self.scale(dense_scores)
        best_indices = np.argsort(combined)[::-1][:top_k]

        return [
            RetrievedChunk(
                title=self.chunks[idx]["title"],
                text=self.chunks[idx]["text"],
                score=float(combined[idx]),
            )
            for idx in best_indices
        ]

## 6. Answer Selection

This version uses Qwen directly as the main answerer, then optionally adds Wikipedia RAG evidence if enough time remains. It scores the next-token probabilities for answer letters A-D, so it does not rely only on parsing generated text.


In [83]:
LETTERS = "ABCD"


def make_context(hits: list[RetrievedChunk], max_chars: int = 2400) -> str:
    blocks = []
    used = 0
    for hit in hits:
        block = f"[{hit.title}] {hit.text}"
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def build_option_lines(options: list) -> str:
    return "\n".join(f"{LETTERS[idx]}. {option_text(option)}" for idx, option in enumerate(options))


def build_user_prompt(question_text: str, options: list, context: str = "") -> str:
    option_lines = build_option_lines(options)
    if context:
        return f"""
Use the evidence when it is relevant, but use your own history knowledge if the evidence is incomplete.
Choose the single best answer.

Evidence:
{context}

Question: {question_text}
{option_lines}

Return only the letter A, B, C, or D.
""".strip()

    return f"""
You are playing a timed multiple-choice history quiz.
Choose the single best answer from the four options.

Question: {question_text}
{option_lines}

Return only the letter A, B, C, or D.
""".strip()


def format_chat_prompt(user_prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are a careful history expert. For multiple-choice questions, answer with exactly one letter: A, B, C, or D.",
        },
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(llm_tokenizer, "apply_chat_template"):
        return llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"


def model_device():
    return next(llm_model.parameters()).device


def parse_llm_choice(generated_text: str, options: list):
    cleaned = generated_text.strip()
    match = re.search(r"\b([A-D])\b", cleaned.upper())
    if match:
        return LETTERS.index(match.group(1))

    lowered = cleaned.lower()
    for idx, option in enumerate(options):
        text = option_text(option).lower()
        if text and text in lowered:
            return idx
    return None


def generate_llm_choice(question_text: str, options: list, context: str = ""):
    prompt = format_chat_prompt(build_user_prompt(question_text, options, context=context))
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=LLM_MAX_INPUT_TOKENS,
    ).to(model_device())

    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    generated = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return parse_llm_choice(generated, options), generated


def letter_logit_scores(question_text: str, options: list, context: str = "") -> list[float]:
    prompt = format_chat_prompt(build_user_prompt(question_text, options, context=context))
    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=LLM_MAX_INPUT_TOKENS,
    ).to(model_device())

    with torch.inference_mode():
        logits = llm_model(**inputs).logits[0, -1]
        log_probs = torch.log_softmax(logits.float(), dim=-1)

    scores = []
    for letter in LETTERS[:len(options)]:
        variants = [letter, " " + letter, "\n" + letter]
        variant_scores = []
        for variant in variants:
            token_ids = llm_tokenizer.encode(variant, add_special_tokens=False)
            if token_ids:
                variant_scores.append(float(log_probs[token_ids[-1]].detach().cpu()))
        scores.append(max(variant_scores) if variant_scores else -100.0)
    return scores


def retrieval_option_scores(question_text: str, options: list, retriever: HybridRetriever, top_k: int = 5):
    scores = []
    all_hits = []
    for option in options:
        text = option_text(option)
        hits = retriever.retrieve(f"{question_text} {text}", top_k=top_k)
        all_hits.append(hits)
        joined = " ".join(hit.text for hit in hits[:3]).lower()
        option_terms = set(tokenize(text))
        joined_terms = set(tokenize(joined))
        overlap = len(option_terms & joined_terms) / max(1, len(option_terms))
        exact = 1.0 if normalize_text(text).lower() in joined else 0.0
        mean_hit = float(np.mean([hit.score for hit in hits[:3]])) if hits else 0.0
        scores.append(mean_hit + 0.20 * overlap + 0.15 * exact)
    return scores, all_hits


def minmax(values: list[float]) -> list[float]:
    if not values:
        return []
    low = min(values)
    high = max(values)
    if abs(high - low) < 1e-8:
        return [0.0 for _ in values]
    return [(value - low) / (high - low) for value in values]


def answer_history_question(question, time_budget: float = 24.0, verbose: bool = True) -> dict:
    start = time.monotonic()
    deadline = start + time_budget
    question_text = question.text if hasattr(question, "text") else question["text"]
    options = question.options if hasattr(question, "options") else question["options"]
    option_texts = [option_text(option) for option in options]

    direct_idx, direct_text = generate_llm_choice(question_text, options, context="")
    direct_scores = letter_logit_scores(question_text, options, context="")

    documents = []
    retrieval_scores = [0.0] * len(options)
    rag_idx = None
    rag_text = "not used"

    if time.monotonic() < deadline - 8:
        documents = collect_history_documents(question_text, option_texts, max_pages=2, deadline=deadline)
        retriever = HybridRetriever(documents, embedding_model)
        retrieval_scores, option_hits = retrieval_option_scores(question_text, options, retriever, top_k=5)
        global_hits = retriever.retrieve(question_text + " " + " ".join(option_texts), top_k=5)

        if global_hits and time.monotonic() < deadline - 5:
            context = make_context(global_hits)
            rag_idx, rag_text = generate_llm_choice(question_text, options, context=context)
    else:
        rag_text = "skipped for timer"

    direct_norm = minmax(direct_scores)
    retrieval_norm = minmax(retrieval_scores)
    combined_scores = [0.0] * len(options)
    for idx in range(len(options)):
        combined_scores[idx] += 0.55 * direct_norm[idx]
        combined_scores[idx] += 0.25 * retrieval_norm[idx]

    if direct_idx is not None:
        combined_scores[direct_idx] += 0.35
    if rag_idx is not None:
        combined_scores[rag_idx] += 0.25

    selected_idx = int(np.argmax(combined_scores))
    selected_option = options[selected_idx]
    elapsed = time.monotonic() - start

    if verbose:
        print(f"Question: {question_text}")
        for idx, option in enumerate(options):
            print(f"  {LETTERS[idx]}. {option_text(option)} | combined={combined_scores[idx]:.3f} direct_logit={direct_scores[idx]:.2f} rag={retrieval_scores[idx]:.3f}")
        print(f"Documents: {[doc['title'] for doc in documents]}")
        print(f"Direct LLM: {LETTERS[direct_idx] if direct_idx is not None else None} | {direct_text}")
        print(f"RAG LLM: {LETTERS[rag_idx] if rag_idx is not None else None} | {rag_text}")
        print(f"Selected: {LETTERS[selected_idx]} -> {option_text(selected_option)}")
        print(f"Elapsed: {elapsed:.2f}s")

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_idx,
        "letter": LETTERS[selected_idx],
        "combined_scores": combined_scores,
        "direct_scores": direct_scores,
        "retrieval_scores": retrieval_scores,
        "direct_choice": LETTERS[direct_idx] if direct_idx is not None else None,
        "direct_output": direct_text,
        "rag_choice": LETTERS[rag_idx] if rag_idx is not None else None,
        "rag_output": rag_text,
        "documents": [doc["title"] for doc in documents],
        "elapsed_seconds": elapsed,
    }


## 7. Small Offline Benchmark

This sanity check verifies that the pipeline works before using a live game attempt.

In [84]:
SAMPLE_HISTORY_QUESTIONS = [
    {
        "text": "Who was the first Roman emperor?",
        "options": [
            {"id": 0, "text": "Julius Caesar"},
            {"id": 1, "text": "Augustus"},
            {"id": 2, "text": "Nero"},
            {"id": 3, "text": "Constantine"},
        ],
        "answer": "Augustus",
    },
    {
        "text": "Which city-state was known for its military society in ancient Greece?",
        "options": [
            {"id": 0, "text": "Athens"},
            {"id": 1, "text": "Sparta"},
            {"id": 2, "text": "Corinth"},
            {"id": 3, "text": "Thebes"},
        ],
        "answer": "Sparta",
    },
    {
        "text": "Who led Carthaginian forces across the Alps during the Second Punic War?",
        "options": [
            {"id": 0, "text": "Hannibal"},
            {"id": 1, "text": "Scipio Africanus"},
            {"id": 2, "text": "Marius"},
            {"id": 3, "text": "Pompey"},
        ],
        "answer": "Hannibal",
    },
]


def run_sample_benchmark(limit=None):
    rows = []
    examples = SAMPLE_HISTORY_QUESTIONS[:limit] if limit else SAMPLE_HISTORY_QUESTIONS
    for idx, sample in enumerate(examples, 1):
        print(f"\nSample {idx}/{len(examples)}")
        result = answer_history_question(sample, time_budget=24.0, verbose=True)
        correct = result["answer_text"].lower() == sample["answer"].lower()
        rows.append({"question": sample["text"], "predicted": result["answer_text"], "gold": sample["answer"], "correct": correct})
        print(f"Correct: {correct}")
    accuracy = sum(row["correct"] for row in rows) / max(1, len(rows))
    print(f"\nSample accuracy: {accuracy:.1%}")
    return rows

benchmark_rows = run_sample_benchmark()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Sample 1/3
Question: Who was the first Roman emperor?
  A. Julius Caesar | combined=0.343 direct_logit=-4.02 rag=0.799
  B. Augustus | combined=1.160 direct_logit=-0.02 rag=0.812
  C. Nero | combined=0.256 direct_logit=-10.52 rag=1.090
  D. Constantine | combined=0.005 direct_logit=-10.64 rag=0.806
Documents: ['The Romans (Doctor Who)', 'Roman']
Direct LLM: B | B
RAG LLM: B | B
Selected: B -> Augustus
Elapsed: 1.53s
Correct: True

Sample 2/3
Question: Which city-state was known for its military society in ancient Greece?
  A. Athens | combined=0.403 direct_logit=-8.13 rag=1.292
  B. Sparta | combined=1.355 direct_logit=-0.00 rag=1.223
  C. Corinth | combined=0.024 direct_logit=-10.75 rag=0.912
  D. Thebes | combined=0.003 direct_logit=-11.25 rag=0.918
Documents: ['Constantine I of Greece', 'Greece']
Direct LLM: B | B
RAG LLM: B | B
Selected: B -> Sparta
Elapsed: 1.82s
Correct: True

Sample 3/3
Question: Who led Carthaginian forces across the Alps during the Second Punic War?
  A. Hann

## 8. Live History Game Runner

Keep `RUN_LIVE_GAME = False` until the setup and benchmark cells work. When ready, set it to `True` and rerun this cell. The runner starts a fresh History game and submits one answer per question.

In [91]:
from datetime import datetime, timezone

RUN_LIVE_GAME = True
RUN_LOG_DIR = Path("/content/gdrive/MyDrive/NLP_assignment/test2_history_runs")


def seconds_available(game) -> float:
    remaining = game.time_remaining
    if remaining is None:
        return QUESTION_SECONDS
    return max(0.0, float(remaining))


def play_history_game(client, comp_id: int = COMP_ID, max_questions=None):
    assert comp_id == 1, "This notebook is configured for the history competition only."
    RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)
    game = client.game.start(competition_id=comp_id)
    run_log = {
        "session_id": game.session_id,
        "competition_id": comp_id,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "questions": [],
    }

    print(f"Started history game. Session ID: {game.session_id}")
    question_count = 0
    correct_count = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            print("No active question returned by server.")
            break

        question_count += 1
        current_level = game.current_level
        print("\n" + "=" * 72)
        print(f"Question {question_count} | Level {current_level} | {seconds_available(game):.1f}s left")
        print("=" * 72)

        budget = min(24.0, max(6.0, seconds_available(game) - 3.0))
        prediction = answer_history_question(question, time_budget=budget, verbose=True)

        print(f"Submitting option id {prediction['answer_id']}: {prediction['answer_text']}")
        result = game.answer(prediction["answer_id"])
        correct_count += int(bool(result.correct))

        run_log["questions"].append(
            {
                "level": current_level,
                "question": question.text,
                "options": [{"id": opt.id, "text": opt.text} for opt in question.options],
                "prediction": prediction,
                "correct": result.correct,
                "timed_out": result.timed_out,
                "game_over": result.game_over,
                "earned_amount": result.earned_amount,
            }
        )

        if result.correct:
            print(f"Correct. Earned so far: {result.earned_amount}")
        elif result.timed_out:
            print(f"Timed out. Final earnings: {result.earned_amount}")
        else:
            print(f"Wrong. Final earnings: {result.earned_amount}")

        if result.game_over:
            break
        if max_questions and question_count >= max_questions:
            print("Manual max_questions limit reached.")
            break

    run_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    run_log["questions_answered"] = question_count
    run_log["correct_count"] = correct_count
    run_log["final_earned_amount"] = game.earned_amount

    log_path = RUN_LOG_DIR / f"history_run_{game.session_id}.json"
    with open(log_path, "w", encoding="utf-8") as handle:
        json.dump(run_log, handle, indent=2)

    print("\nGame summary")
    print(f"Questions answered: {question_count}")
    print(f"Correct answers: {correct_count}")
    print(f"Final earnings: {game.earned_amount}")
    print(f"Run log: {log_path}")
    return game, run_log


if RUN_LIVE_GAME:
    final_game, final_log = play_history_game(client, comp_id=COMP_ID)
else:
    print("Set RUN_LIVE_GAME = True and rerun this cell to play the live history game.")


Started history game. Session ID: 37111

Question 1 | Level 1 | 29.9s left
Wikipedia search skipped for 'What Roman Roman Kingdom Republic': HTTP Error 429: Too Many Requests
Wikipedia search skipped for 'describes body roman citizens kingdom republic empire': HTTP Error 429: Too Many Requests
Wikipedia search skipped for 'describes body roman citizens kingdom republic empire praetorian guard senate': HTTP Error 429: Too Many Requests
Wikipedia search skipped for 'What term describes the body of Roman citizens during the Roman Kingdom, Republic, and Empire?': HTTP Error 429: Too Many Requests
Question: What term describes the body of Roman citizens during the Roman Kingdom, Republic, and Empire?
  A. The Praetorian Guard | combined=0.073 direct_logit=-9.75 rag=0.000
  B. The Senate | combined=0.000 direct_logit=-11.25 rag=0.000
  C. The Roman people | combined=0.900 direct_logit=-0.00 rag=0.000
  D. The Legion | combined=0.037 direct_logit=-10.50 rag=0.000
Documents: []
Direct LLM: C |

## 9. Presentation Notes

Architecture: local open-weight model plus RAG. Retrieval is kept small because every live question has a 30 second timer.

Strengths:

- good for factual history questions with names, battles, empires, dates, and institutions;
- less hallucination-prone than a plain local LLM because it sees retrieved evidence;
- retrieval-only fallback still works when generation is too slow.

Weaknesses:

- obscure questions may not retrieve the right page;
- if all options appear in the same broad article, option ranking can be noisy;
- the 30 second timer limits the amount of retrieved context.

Possible extensions:

- compare `flan-t5-base` and `flan-t5-large`;
- pre-cache a local history corpus before live play;
- ensemble this model with another local QA or NLI model.